# SAE latent space: UMAP / t-SNE

## Quick recap: your SAE architecture

- **Input**: For each residue pair `(i, j)`, a **384-D context** = concat of pair embedding `(i,j)`, row diagonal `(i,i)`, column diagonal `(j,j)` from AlphaFold **pair block 47** `(L, L, 128)`.
- **Encoder**: `Linear(384 → 4096)` → pre-activations `z`.
- **Sparsity**: **Adaptive top-k softmax** (target mass τ): each token keeps only the smallest `k` latents needed so the descending-softmax CDF reaches τ; then **ReLU(z) × mask**.
- **Decoder**: `Linear(4096 → 128)` → **tanh** reconstruction of the 128-D pair channel.
- **Training**: MSE reconstruction + entropy penalty on full softmax `p(z)` to encourage sharp, sparse codes.

**Your TM-scores (mean ≈ 0.89)** show that structures rebuilt from SAE reconstructions usually align well to natives. Values **> ~0.8** are typically strong for same protein; **~0.5–0.7** often still same fold but more distortion; **< ~0.2** is random/unrelated. A few chains in your list (~0.55–0.67) are worth eyeballing in a viewer.

## What you can do in latent space

- **Per-protein embedding**: pool sparse latents `(L² × 4096)` → one vector (e.g. mean) → cluster proteins by interaction-map “style” or quality.
- **Dictionary / features**: inspect which latent dimensions activate for which contacts; correlate with secondary structure, distance bins, or TM-score.
- **Steering / denoising**: interpolate or ablate latents before decoding to see which factors change the pair map (and downstream structure).
- **Outliers**: proteins far from the UMAP mass might be harder reconstructions (consistent with lower TM-score).

This notebook builds **one vector per protein** (default: mean-pooled sparse latents), optional **PCA**, then **UMAP** (if `umap-learn` installed) else **t-SNE**, and plots points colored by **TM-score** when `tm_scores.json` exists.

**Data layout**: Nested **`proteins_layer47/<id>/<id>_pair_block_47.npy`** (same pattern under `CompleteProteins/…` if you use that name), or **flat** `*.npy` in one folder. If no `PROTEIN_DIR` env is set, the first existing folder among `proteins_layer47`, `CompleteProteins` under `AUTOENCODER_ROOT` is used; the loader retries the other if the first has no tensors.

**Env overrides**: `AUTOENCODER_ROOT`, `PROTEIN_DIR`, **`TOKEN_SAE_CHECKPOINT`** (exact `.pt` file), **`TOKEN_SAE_TRAINING_OUTPUT`** (folder that contains `token_sae_best.pt`; same as train notebook’s output dir), **`SCRATCH`** (if weights live under `$SCRATCH/AlphaFold_Autoencoder/proteins_layer47/token_sae_notebook_output/`), `TM_SCORES_JSON`, `PAIR_LAYER`.

Checkpoints are auto-picked from `PROTEIN_DIR/token_sae_notebook_output/` (matches **`train_token_sae.ipynb`**), then the same path under `AUTOENCODER_ROOT/proteins_layer47/`, then `$SCRATCH/.../proteins_layer47/` — **`token_sae_best.pt`** before **`token_sae_final.pt`**.

In [ ]:
from __future__ import annotations

import json
import os
import sys
from pathlib import Path

import numpy as np
import torch
from torch.utils.data import DataLoader


def _find_script_dir() -> Path:
    p = Path.cwd().resolve()
    for _ in range(10):
        if (p / "sparse_autoencoder.py").is_file():
            return p
        p = p.parent
    return Path.cwd().resolve()


SCRIPT_DIR = _find_script_dir()
AUTOENCODER_ROOT = Path(os.environ.get("AUTOENCODER_ROOT", str(SCRIPT_DIR.parent))).resolve()

_pd_env = os.environ.get("PROTEIN_DIR", "").strip()
if _pd_env:
    PROTEIN_DIR = Path(_pd_env).resolve()
else:
    PROTEIN_DIR = None
    for _cand in (AUTOENCODER_ROOT / "proteins_layer47", AUTOENCODER_ROOT / "CompleteProteins"):
        if _cand.is_dir():
            PROTEIN_DIR = _cand.resolve()
            break
    if PROTEIN_DIR is None:
        PROTEIN_DIR = (AUTOENCODER_ROOT / "proteins_layer47").resolve()
def _resolve_token_sae_checkpoint() -> Path:
    """Prefer train_token_sae.ipynb default: .../proteins_layer47/token_sae_notebook_output/*.pt"""
    env_ck = os.environ.get("TOKEN_SAE_CHECKPOINT", "").strip()
    if env_ck:
        return Path(env_ck).resolve()

    out_dir_env = os.environ.get("TOKEN_SAE_TRAINING_OUTPUT", "").strip()
    if out_dir_env:
        d = Path(out_dir_env).expanduser().resolve()
        for name in ("token_sae_best.pt", "token_sae_final.pt"):
            p = d / name
            if p.is_file():
                return p

    tiebreak = ("token_sae_best.pt", "token_sae_final.pt")
    candidates: list[Path] = []

    def add_notebook_output_under(proteins_layer47_root: Path) -> None:
        d = proteins_layer47_root / "token_sae_notebook_output"
        for name in tiebreak:
            candidates.append(d / name)

    if PROTEIN_DIR.is_dir() and PROTEIN_DIR.name == "proteins_layer47":
        add_notebook_output_under(PROTEIN_DIR)
    add_notebook_output_under(AUTOENCODER_ROOT / "proteins_layer47")

    scratch = os.environ.get("SCRATCH", "").strip()
    if scratch:
        add_notebook_output_under(Path(scratch) / "AlphaFold_Autoencoder" / "proteins_layer47")

    fallback = [
        SCRIPT_DIR / "token_sae_best.pt",
        SCRIPT_DIR / "checkpoints" / "token_sae_best.pt",
        SCRIPT_DIR.parent / "token_sae_best.pt",
        SCRIPT_DIR.parent / "checkpoints" / "token_sae_best.pt",
        SCRIPT_DIR.parent.parent / "token_sae_best.pt",
        AUTOENCODER_ROOT / "token_sae_best.pt",
        AUTOENCODER_ROOT / "original_SAE" / "token_sae_best.pt",
        AUTOENCODER_ROOT / "checkpoints" / "token_sae_best.pt",
    ]

    for p in candidates + fallback:
        if p.is_file():
            return p.resolve()
    for root in (
        SCRIPT_DIR,
        SCRIPT_DIR.parent,
        SCRIPT_DIR.parent.parent,
        AUTOENCODER_ROOT,
        AUTOENCODER_ROOT / "original_SAE",
    ):
        if not root.is_dir():
            continue
        for pat in ("token_sae*.pt", "*token*sae*.pt"):
            hits = sorted(root.glob(pat))
            if hits:
                return hits[0].resolve()
    return (AUTOENCODER_ROOT / "proteins_layer47" / "token_sae_notebook_output" / "token_sae_best.pt").resolve()


CHECKPOINT = _resolve_token_sae_checkpoint()

TM_SCORES_JSON = Path(
    os.environ.get("TM_SCORES_JSON", str(AUTOENCODER_ROOT / "tm_scores" / "tm_scores.json"))
).resolve()

PAIR_LAYER = int(os.environ.get("PAIR_LAYER", "47"))
D_LATENT = int(os.environ.get("TOKEN_SAE_D_LATENT", "4096"))
TAU = float(os.environ.get("TOKEN_SAE_TAU", "0.90"))

if str(SCRIPT_DIR) not in sys.path:
    sys.path.insert(0, str(SCRIPT_DIR))

from sparse_autoencoder import (
    ContextualPairDataset,
    ContextualTokenSAE,
    pack_context_collate,
)

print("SCRIPT_DIR:", SCRIPT_DIR)
print("AUTOENCODER_ROOT:", AUTOENCODER_ROOT)
print("PROTEIN_DIR:", PROTEIN_DIR)
print("CHECKPOINT:", CHECKPOINT)
print("TM_SCORES_JSON:", TM_SCORES_JSON)

In [ ]:
def discover_pair_paths(protein_dir: Path, layer: int = 47) -> list[str]:
    """Nested layout (<id>/<id>_pair_block_L.npy) or flat (*.npy in one folder)."""
    paths: list[str] = []
    if not protein_dir.is_dir():
        return paths

    for item in sorted(protein_dir.iterdir()):
        if not item.is_dir():
            continue
        name = item.name
        p = item / f"{name}_pair_block_{layer}.npy"
        if p.is_file():
            paths.append(str(p))
    if paths:
        return paths

    flat = sorted(protein_dir.glob(f"*_pair_block_{layer}.npy"))
    if flat:
        return [str(p) for p in flat]

    for p in sorted(protein_dir.glob("*.npy")):
        try:
            a = np.load(p, mmap_mode="r")
            if a.ndim == 3 and a.shape[2] == 128:
                paths.append(str(p))
        except Exception:
            continue
    return paths


def protein_id_from_npy_path(path: str, layer: int) -> str:
    """Match tm_scores keys (e.g. 6z01_A) from nested dir or flat filename."""
    p = Path(path).resolve()
    if p.parent.name and p.name == f"{p.parent.name}_pair_block_{layer}.npy":
        return p.parent.name
    stem = p.stem
    suf = f"_pair_block_{layer}"
    if stem.endswith(suf):
        return stem[: -len(suf)]
    return stem


paths = discover_pair_paths(PROTEIN_DIR, PAIR_LAYER)
if not paths:
    for _alt in (AUTOENCODER_ROOT / "proteins_layer47", AUTOENCODER_ROOT / "CompleteProteins"):
        if _alt.is_dir() and _alt.resolve() != PROTEIN_DIR.resolve():
            paths = discover_pair_paths(_alt, PAIR_LAYER)
            if paths:
                print(f"No pair tensors under {PROTEIN_DIR}; using {_alt} ({len(paths)} files).")
                PROTEIN_DIR = _alt.resolve()
                break
if not paths:
    raise FileNotFoundError(
        f"No pair .npy under {PROTEIN_DIR} (nested <id>/<id>_pair_block_{PAIR_LAYER}.npy, "
        f"or flat *_pair_block_{PAIR_LAYER}.npy, or flat *.npy shaped L×L×128). "
        f"export PROTEIN_DIR=/path/to/proteins_layer47"
    )
if not CHECKPOINT.is_file():
    raise FileNotFoundError(
        "Checkpoint not found. Place token_sae_best.pt next to this notebook, or run:\n"
        f"  export TOKEN_SAE_CHECKPOINT=/path/to/your/token_sae_best.pt\n"
        f"Last tried: {CHECKPOINT}\n"
        f"Searched under: {SCRIPT_DIR} and {AUTOENCODER_ROOT} (incl. original_SAE/)."
    )

dataset = ContextualPairDataset(paths, normalize=True)
loader = DataLoader(
    dataset,
    batch_size=1,
    shuffle=False,
    collate_fn=pack_context_collate,
)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = ContextualTokenSAE(
    d_context_in=384,
    d_latent=D_LATENT,
    d_recon_out=128,
    tau=TAU,
).to(device)
try:
    _sd = torch.load(CHECKPOINT, map_location=device, weights_only=True)
except TypeError:
    _sd = torch.load(CHECKPOINT, map_location=device)
model.load_state_dict(_sd)
model.eval()

protein_ids: list[str] = []
embeddings: list[np.ndarray] = []
sparsity: list[float] = []

with torch.no_grad():
    for i, (packed_context, _tgt, _shapes) in enumerate(loader):
        packed_context = packed_context.to(device)
        _recon, _p, latents = model(packed_context)
        z = latents.float().cpu().numpy()
        protein_ids.append(protein_id_from_npy_path(paths[i], PAIR_LAYER))
        embeddings.append(z.mean(axis=0))
        sparsity.append(float((np.abs(z) > 1e-6).mean()))

X = np.stack(embeddings, axis=0)
print(f"Proteins: {len(protein_ids)}, embedding dim: {X.shape[1]}")

In [ ]:
from sklearn.decomposition import PCA
from sklearn.manifold import TSNE
from sklearn.preprocessing import StandardScaler

try:
    import umap

    _has_umap = True
except ImportError:
    _has_umap = False
    print("umap-learn not installed; using t-SNE. Install: pip install umap-learn")

PCA_DIM = 50
RANDOM_STATE = 42

scaler = StandardScaler()
Xz = scaler.fit_transform(X)
if Xz.shape[1] > PCA_DIM:
    Xp = PCA(n_components=PCA_DIM, random_state=RANDOM_STATE).fit_transform(Xz)
else:
    Xp = Xz

if _has_umap:
    reducer = umap.UMAP(
        n_components=2,
        n_neighbors=min(15, max(2, len(protein_ids) - 1)),
        min_dist=0.1,
        metric="cosine",
        random_state=RANDOM_STATE,
    )
    XY = reducer.fit_transform(Xp)
    method_name = "UMAP"
else:
    _perp = min(30, max(5, len(protein_ids) // 4))
    _tsne_kw = dict(
        n_components=2,
        init="pca",
        perplexity=_perp,
        random_state=RANDOM_STATE,
    )
    try:
        XY = TSNE(learning_rate="auto", **_tsne_kw).fit_transform(Xp)
    except (TypeError, ValueError):
        XY = TSNE(learning_rate=200, **_tsne_kw).fit_transform(Xp)
    method_name = "t-SNE"

print(method_name, XY.shape)

In [ ]:
import matplotlib.pyplot as plt

tm_by_id: dict[str, float] = {}
if TM_SCORES_JSON.is_file():
    with open(TM_SCORES_JSON) as f:
        data = json.load(f)
    tm_by_id = {k: float(v) for k, v in data.get("individual_scores", {}).items()}

colors = np.array([tm_by_id.get(pid, np.nan) for pid in protein_ids])
has_tm = np.isfinite(colors)

fig, ax = plt.subplots(figsize=(8, 6))
if has_tm.any():
    sc = ax.scatter(XY[has_tm, 0], XY[has_tm, 1], c=colors[has_tm], cmap="viridis", s=28, alpha=0.85)
    plt.colorbar(sc, ax=ax, label="TM-score")
    miss = ~has_tm
    if miss.any():
        ax.scatter(XY[miss, 0], XY[miss, 1], c="lightgray", s=22, alpha=0.6, label="no TM-score")
        ax.legend()
else:
    ax.scatter(XY[:, 0], XY[:, 1], c="steelblue", s=28, alpha=0.85)
    ax.text(
        0.02,
        0.98,
        f"No {TM_SCORES_JSON.name}; run tm_scores.ipynb first",
        transform=ax.transAxes,
        va="top",
        fontsize=9,
    )

ax.set_title(f"Proteins in SAE latent space ({method_name}); mean-pooled sparse codes")
ax.set_xlabel(f"{method_name} 1")
ax.set_ylabel(f"{method_name} 2")
plt.tight_layout()
plt.show()

fig2, ax2 = plt.subplots(figsize=(6, 4))
ax2.scatter(
    [tm_by_id.get(pid, np.nan) for pid in protein_ids],
    sparsity,
    c="darkorange",
    alpha=0.7,
    edgecolors="k",
    linewidths=0.2,
)
ax2.set_xlabel("TM-score (if available)")
ax2.set_ylabel("Mean latent sparsity (fraction |z|>1e-6)")
ax2.set_title("Reconstruction quality vs activation density")
plt.tight_layout()
plt.show()

In [ ]:
out_npz = AUTOENCODER_ROOT / "latent_analysis" / "protein_latent_mean_pooled.npz"
out_npz.parent.mkdir(parents=True, exist_ok=True)
np.savez(
    out_npz,
    protein_ids=np.array(protein_ids),
    X_latent_mean=X,
    umap_or_tsne=XY,
    method=np.array([method_name]),
    sparsity=np.array(sparsity),
)
print("Saved:", out_npz)